# 01 — Pipeline ETL do CPGF

Este notebook transforma os CSVs de 2023–2025 em pandas e carrega o esquema estrela em SQLite. **Uma linha da origem corresponde a uma linha da fato**, inclusive repetições exatas.

A carga é **completa**: constrói e valida um banco temporário e só então substitui o destino. As SKs sequenciais podem mudar quando a entrada muda; fato e dimensões são sempre reconstruídas juntas. Execute **Reiniciar kernel e executar tudo**, na ordem.

Toda a lógica ETL está aqui. O contrato das tabelas permanece em `scripts/sql/ddl/schema.sql`. Não há download automático, alteração dos CSVs nem implementação do ELT.

## 1. Ambiente e configuração
Abra este notebook a partir de um clone do projeto, localmente ou no Colab. Dependência: pandas (`%pip install pandas`, caso ainda não esteja instalado no kernel). No Colab, clone o repositório e ajuste `RAIZ_CONFIGURADA` para a pasta do clone. Os CSVs completos devem ter sido gerados por `scripts/extract_cpgf.py`.

O modo padrão exige os três arquivos completos. `MODO_AMOSTRA=True` usa somente as amostras e grava **outro banco**, evitando substituir o resultado completo. As amostras não demonstram cobertura de todos os meses nem de sigilo.

In [ ]:
from pathlib import Path
from decimal import Decimal
import hashlib
import json
import re
import sqlite3
import tempfile
import time

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Permite validar as células fora do Jupyter.

RAIZ_CONFIGURADA = None  # Exemplo no Colab: Path('/content/projeto-integracao-dw')
MODO_AMOSTRA = False
ANOS = (2023, 2024, 2025)


def localizar_raiz(configurada=None):
    candidatos = [Path(configurada)] if configurada is not None else [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / 'scripts/sql/ddl/schema.sql').is_file():
            return pasta.resolve()
    raise FileNotFoundError('Configure RAIZ_CONFIGURADA com a pasta do repositório.')


RAIZ = localizar_raiz(RAIZ_CONFIGURADA)
ARQUIVOS = [
    RAIZ / 'data/raw' / (f'amostras/{ano}_amostra.csv' if MODO_AMOSTRA else f'{ano}_CPGF.csv')
    for ano in ANOS
]
DESTINO = RAIZ / 'data/database' / ('cpgf_dw_amostra.sqlite' if MODO_AMOSTRA else 'cpgf_dw.sqlite')
DDL = RAIZ / 'scripts/sql/ddl/schema.sql'
INICIO = time.perf_counter()
print('pandas:', pd.__version__)
print('Modo:', 'amostra' if MODO_AMOSTRA else 'completo', '| Destino:', DESTINO)

## 2. Leitura e rastreabilidade
Os CSVs anuais produzidos pelo extrator são UTF-8, separados por `;`. Lemos tudo como texto para preservar zeros à esquerda e máscaras. A origem e a posição do registro acompanham a transformação, mas não alteram o esquema final.

O manifesto mostra tamanho, hash e quantidade de registros efetivamente lidos. Quando os metadados da extração existem, conferimos as quantidades por mês. Não fixamos a contagem do relatório antigo: o portal pode revisar arquivos.

In [ ]:
COLUNAS = {
    'CÓDIGO ÓRGÃO SUPERIOR': 'codigo_orgao_superior',
    'NOME ÓRGÃO SUPERIOR': 'nome_orgao_superior',
    'CÓDIGO ÓRGÃO': 'codigo_orgao',
    'NOME ÓRGÃO': 'nome_orgao',
    'CÓDIGO UNIDADE GESTORA': 'codigo_unidade_gestora',
    'NOME UNIDADE GESTORA': 'nome_unidade_gestora',
    'ANO EXTRATO': 'ano_extrato',
    'MÊS EXTRATO': 'mes_extrato',
    'CPF PORTADOR': 'cpf_mascarado',
    'NOME PORTADOR': 'nome_portador',
    'CNPJ OU CPF FAVORECIDO': 'documento',
    'NOME FAVORECIDO': 'nome_favorecido',
    'TRANSAÇÃO': 'descricao_original',
    'DATA TRANSAÇÃO': 'data_original',
    'VALOR TRANSAÇÃO': 'valor_original',
}


def exigir(condicao, mensagem):
    if not bool(condicao):
        raise ValueError(mensagem)


def ler_entradas(arquivos):
    ausentes = [str(p) for p in arquivos if not p.is_file()]
    if ausentes:
        raise FileNotFoundError('Arquivos ausentes: ' + ', '.join(ausentes))
    partes, manifesto = [], []
    for arquivo in arquivos:
        df = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
        exigir(list(df.columns) == list(COLUNAS), f'{arquivo.name}: cabeçalho diferente do esperado.')
        exigir(len(df) > 0, f'{arquivo.name}: arquivo vazio.')
        df = df.rename(columns=COLUNAS)
        df = df.apply(lambda s: s.str.strip())
        df['arquivo_origem'] = arquivo.name
        df['registro_origem'] = range(1, len(df) + 1)
        exigir(df['ano_extrato'].eq(arquivo.name[:4]).all(), f'{arquivo.name}: ano de extrato divergente.')
        partes.append(df)
        manifesto.append({
            'arquivo': arquivo.name, 'registros': len(df),
            'bytes': arquivo.stat().st_size,
            'sha256': hashlib.sha256(arquivo.read_bytes()).hexdigest(),
        })
    return pd.concat(partes, ignore_index=True), pd.DataFrame(manifesto)


def conferir_metadados(dados, caminho):
    if not caminho.is_file():
        print('Metadados ausentes: contagens da extração não foram conferidas.')
        return
    metadados = json.loads(caminho.read_text(encoding='utf-8'))
    registros = [m for m in metadados if int(m['referencia'][:4]) in ANOS]
    esperado = {m['referencia']: m['qtd_registros'] for m in registros}
    periodos = {f'{ano}{mes:02d}' for ano in ANOS for mes in range(1, 13)}
    exigir(len(registros) == len(periodos) and set(esperado) == periodos,
           'Metadados devem conter uma entrada para cada mês solicitado.')
    obtido = (dados['ano_extrato'] + dados['mes_extrato'].str.zfill(2)).value_counts().to_dict()
    exigir(all(obtido.get(p, 0) == n for p, n in esperado.items()) and set(obtido) <= periodos,
           'Quantidades por mês divergem dos metadados da extração.')


brutos, manifesto = ler_entradas(ARQUIVOS)
if not MODO_AMOSTRA:
    conferir_metadados(brutos, RAIZ / 'data/raw/metadados.json')
display(manifesto)
print('Registros de entrada:', len(brutos))

## 3. Limpeza e regras de negócio

- Valores: aceitar formato brasileiro, com `R$` opcional; preservar negativos e converter para **centavos inteiros**. Valor inválido interrompe o processamento.
- Datas: interpretar estritamente `dd/mm/aaaa`. Data ausente recebe `-1`; data preenchida inválida gera erro. O extrato usa o primeiro dia do mês e deve ter ano e mês válidos.
- Sigilo: usar a descrição administrativa exata ou documento `-11`, nunca procurar a palavra em qualquer nome. Data e portador sigilosos recebem `-1`. Ausência parcial de identificação do portador é interrompida para análise.
- Documentos administrativos `-1`, `-2`, `-11` continuam sendo documentos da fonte, distintos das SKs.

Mantemos `DECIMAL(15,2)` no SQLite para respeitar o DDL. Ele não garante armazenamento decimal exato; por isso a reconciliação é feita em centavos, e consultas monetárias no DW devem seguir a convenção documentada de arredondamento. Não há conversão cambial.

In [ ]:
MARCADORES_SIGILO = {'sigiloso', 'informações protegidas por sigilo'}


def converter_centavos(serie):
    texto = serie.str.replace(r'^R\$\s*', '', regex=True).str.strip()
    valido = texto.str.fullmatch(r'-?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+),[0-9]{2}', na=False)
    exigir(valido.all(), f'Valores monetários inválidos nos índices: {serie.index[~valido].tolist()[:10]}')
    numeros = texto.str.replace('.', '', regex=False).str.replace(',', '', regex=False).map(int)
    exigir(numeros.abs().le(999999999999999).all(), 'Valor fora da precisão prevista em DECIMAL(15,2).')
    return numeros.astype('int64')


def transformar(brutos):
    dados = brutos.copy()
    obrigatorias = [c for c in COLUNAS.values() if c not in {'data_original', 'cpf_mascarado', 'nome_portador'}]
    exigir(dados[obrigatorias].ne('').all().all(), 'Há campos obrigatórios vazios na entrada.')
    dados['valor_centavos'] = converter_centavos(dados['valor_original'])
    dados['sigilosa'] = (
        dados['descricao_original'].str.casefold().isin(MARCADORES_SIGILO)
        | dados['nome_portador'].str.casefold().isin(MARCADORES_SIGILO)
        | dados['nome_favorecido'].str.casefold().isin(MARCADORES_SIGILO)
        | dados['documento'].eq('-11')
    )
    datas = pd.to_datetime(dados['data_original'], format='%d/%m/%Y', errors='coerce')
    preenchidas = dados['data_original'].ne('')
    formato = dados['data_original'].str.fullmatch(r'[0-9]{2}/[0-9]{2}/[0-9]{4}')
    invalidas = preenchidas & (datas.isna() | ~formato)
    exigir(~invalidas.any(), f'Datas inválidas nos índices: {dados.index[invalidas].tolist()[:10]}')
    ano = dados['ano_extrato']
    mes = dados['mes_extrato']
    exigir(ano.str.fullmatch(r'[0-9]{4}').all() and mes.str.fullmatch(r'[0-9]{1,2}').all(),
           'Ano ou mês do extrato fora do formato esperado.')
    exigir(ano.astype(int).isin(ANOS).all() and mes.astype(int).between(1, 12).all(),
           'Ano ou mês do extrato fora do intervalo esperado.')
    dados['data_transacao'] = datas.mask(dados['sigilosa'])
    dados['data_extrato'] = pd.to_datetime(ano + '-' + mes.str.zfill(2) + '-01', format='%Y-%m-%d')
    dados['id_data_transacao'] = dados['data_transacao'].dt.strftime('%Y%m%d').fillna('-1').astype('int64')
    dados['id_mes_extrato'] = dados['data_extrato'].dt.strftime('%Y%m%d').astype('int64')
    cpf_vazio = dados['cpf_mascarado'].eq('')
    nome_vazio = dados['nome_portador'].eq('')
    exigir(~((cpf_vazio ^ nome_vazio) & ~dados['sigilosa']).any(),
           'Portador parcialmente identificado fora do sigilo: revisar antes de carregar.')
    dados['portador_especial'] = dados['sigilosa'] | (cpf_vazio & nome_vazio)
    return dados


dados = transformar(brutos)
display(pd.Series({
    'transacoes': len(dados),
    'sigilosas': int(dados['sigilosa'].sum()),
    'data_especial': int(dados['id_data_transacao'].eq(-1).sum()),
    'portador_especial': int(dados['portador_especial'].sum()),
    'total_centavos': int(dados['valor_centavos'].sum()),
}, name='Diagnóstico'))

## 4. Construção das dimensões e das surrogate keys
Deduplicamos **somente dimensões** pelas chaves naturais previstas no DDL. Ordenamos os membros e geramos SKs positivas em pandas. Se uma chave natural tiver atributos conflitantes (por exemplo, os mesmos códigos de órgão com nomes diferentes), interrompemos para decidir a regra de negócio.

O calendário cobre anos completos, incluindo datas de transação anteriores a 2023. `DIM_TEMPO` usa `YYYYMMDD`, uma chave derivada da data; inclui um membro `-1`, assim como `DIM_PORTADOR`.

A classificação de tipos é uma tabela explícita e revisável. `INT$` segue a categoria `USD` prevista na modelagem; isso descreve a modalidade e **não comprova a moeda de liquidação nem converte valores**. Descrições não mapeadas permanecem em `Outros` / `Não Aplicável` e aparecem no diagnóstico para revisão.

In [ ]:
CHAVE_ORGAO = ['codigo_unidade_gestora', 'codigo_orgao', 'codigo_orgao_superior']
CHAVE_PORTADOR = ['cpf_mascarado', 'nome_portador']
CHAVE_FAVORECIDO = ['documento', 'nome_favorecido']


def criar_dimensao(base, chave, atributos, id_coluna):
    dim = base[chave + atributos].drop_duplicates().copy()
    exigir(~dim.duplicated(chave).any(), f'{id_coluna}: atributos conflitantes para a mesma chave natural.')
    dim = dim.sort_values(chave, kind='stable').reset_index(drop=True)
    dim.insert(0, id_coluna, range(1, len(dim) + 1))
    return dim


def classificar_documento(documento):
    if documento in {'-1', '-2', '-11'}:
        return 'Especial'
    if re.fullmatch(r'\*{3}\.[0-9]{3}\.[0-9]{3}-\*{2}', documento):
        return 'PF'
    if not re.fullmatch(r'[0-9./-]+', documento):
        raise ValueError('Documento fora dos formatos previstos; revisar a origem.')
    digitos = re.sub(r'[./-]', '', documento)
    if len(digitos) == 14:
        return 'PJ'
    if len(digitos) == 11:
        return 'PF'
    raise ValueError('Documento fora dos comprimentos previstos; revisar a origem.')


TIPOS = {
    'COMPRA A/V - R$ - APRES': ('Compra', 'BRL', 'Cartão / Fatura'),
    'COMPRA A/V - INT$ - APRES': ('Compra', 'USD', 'Cartão / Fatura'),
    'SAQUE CASH/ATM BB': ('Saque', 'BRL', 'Espécie / Autoatendimento'),
    'SAQUE - R$ - APRES': ('Saque', 'BRL', 'Espécie / Autoatendimento'),
    'SAQUE - INT$ - APRES': ('Saque', 'USD', 'Espécie / Autoatendimento'),
    'Informações protegidas por sigilo': ('Sigiloso', 'Não Aplicável', 'Sigiloso'),
}


def criar_calendario(dados):
    limites = pd.concat([dados['data_transacao'], dados['data_extrato']]).dropna()
    inicio = min(min(ANOS), int(limites.dt.year.min()))
    fim = max(max(ANOS), int(limites.dt.year.max()))
    datas = pd.Series(pd.date_range(f'{inicio}-01-01', f'{fim}-12-31'))
    meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
             'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']
    dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo']
    calendario = pd.DataFrame({
        'id_tempo': datas.dt.strftime('%Y%m%d').astype('int64'),
        'data': datas.dt.strftime('%Y-%m-%d'),
        'dia': datas.dt.day, 'mes': datas.dt.month,
        'nome_mes': datas.dt.month.map(dict(enumerate(meses, 1))),
        'trimestre': datas.dt.quarter, 'semestre': (datas.dt.month - 1) // 6 + 1,
        'ano': datas.dt.year,
        'dia_semana': datas.dt.dayofweek.map(dict(enumerate(dias))),
        'fim_de_semana': datas.dt.dayofweek.ge(5).astype(int),
    })
    especial = {col: None for col in calendario.columns}
    especial.update(id_tempo=-1, nome_mes='Não Informado / Sigiloso', dia_semana='Não Informado')
    calendario = pd.concat([pd.DataFrame([especial]), calendario], ignore_index=True)
    for col in ['id_tempo', 'dia', 'mes', 'trimestre', 'semestre', 'ano', 'fim_de_semana']:
        calendario[col] = calendario[col].astype('Int64')
    return calendario


def construir_dimensoes(dados):
    orgao = criar_dimensao(dados, CHAVE_ORGAO,
                          ['nome_unidade_gestora', 'nome_orgao', 'nome_orgao_superior'], 'id_orgao')
    portador = criar_dimensao(dados.loc[~dados['portador_especial']], CHAVE_PORTADOR, [], 'id_portador')
    portador = pd.concat([pd.DataFrame([{
        'id_portador': -1, 'cpf_mascarado': 'SIGILOSO',
        'nome_portador': 'Portador Sigiloso / Não Identificado',
    }]), portador], ignore_index=True)
    favorecido = criar_dimensao(dados, CHAVE_FAVORECIDO, [], 'id_favorecido')
    favorecido['tipo_pessoa'] = favorecido['documento'].map(classificar_documento)
    tipo = criar_dimensao(dados, ['descricao_original'], [], 'id_tipo_transacao')
    classificacao = tipo['descricao_original'].map(lambda s: TIPOS.get(s, ('Outros', 'Não Aplicável', 'Outros')))
    tipo[['categoria', 'moeda', 'canal']] = pd.DataFrame(classificacao.tolist(), index=tipo.index)
    return {'DIM_TEMPO': criar_calendario(dados), 'DIM_ORGAO': orgao,
            'DIM_PORTADOR': portador, 'DIM_FAVORECIDO': favorecido, 'DIM_TIPO_TRANSACAO': tipo}


dimensoes = construir_dimensoes(dados)
display(pd.Series({nome: len(df) for nome, df in dimensoes.items()}, name='Membros por dimensão'))
display(dimensoes['DIM_TIPO_TRANSACAO'])

## 5. Construção da fato
Cada junção é validada como **muitos para um**: várias transações podem apontar para o mesmo membro, mas a dimensão não pode multiplicar as linhas da fato. As SKs das dimensões são geradas antes da carga. O `id_transacao` é gerado pelo SQLite, conforme o dicionário.

Centavos, órgão, mês e posição de origem permanecem disponíveis em memória para conferir o resultado. A fato não recebe dados brutos nem elimina linhas repetidas.

In [ ]:
COLUNAS_FATO = ['id_data_transacao', 'id_mes_extrato', 'id_orgao', 'id_portador',
                'id_favorecido', 'id_tipo_transacao', 'valor_transacao', 'quantidade_transacoes']


def juntar_dimensao(base, dim, chave, id_coluna):
    resultado = base.merge(dim[chave + [id_coluna]], on=chave, how='left', validate='many_to_one', sort=False)
    exigir(len(resultado) == len(base), f'{id_coluna}: junção alterou a quantidade de registros.')
    exigir(resultado[id_coluna].notna().all(), f'{id_coluna}: há membros sem correspondência.')
    resultado[id_coluna] = resultado[id_coluna].astype('int64')
    return resultado


def construir_fato(dados, dimensoes):
    base = dados.copy()
    base['_ordem'] = range(len(base))
    base.loc[base['portador_especial'], 'cpf_mascarado'] = 'SIGILOSO'
    base.loc[base['portador_especial'], 'nome_portador'] = 'Portador Sigiloso / Não Identificado'
    for tabela, chave, id_coluna in [
        ('DIM_ORGAO', CHAVE_ORGAO, 'id_orgao'),
        ('DIM_PORTADOR', CHAVE_PORTADOR, 'id_portador'),
        ('DIM_FAVORECIDO', CHAVE_FAVORECIDO, 'id_favorecido'),
        ('DIM_TIPO_TRANSACAO', ['descricao_original'], 'id_tipo_transacao'),
    ]:
        base = juntar_dimensao(base, dimensoes[tabela], chave, id_coluna)
    base = base.sort_values('_ordem').reset_index(drop=True)
    # Texto decimal exato na fronteira de carga; SQLite aplica afinidade NUMERIC.
    base['valor_transacao'] = base['valor_centavos'].map(lambda n: format(Decimal(int(n)) / 100, '.2f'))
    base['quantidade_transacoes'] = 1
    fato = base[COLUNAS_FATO].copy()
    exigir(len(fato) == len(dados), 'Quantidade da fato difere da origem.')
    exigir(base['valor_centavos'].equals(dados['valor_centavos'].reset_index(drop=True)),
           'Valores ou ordem das transações foram alterados pelas junções.')
    for coluna in ['id_data_transacao', 'id_mes_extrato']:
        exigir(fato[coluna].isin(dimensoes['DIM_TEMPO']['id_tempo']).all(), 'Calendário incompleto.')
    exigir(fato.loc[dados['portador_especial'].to_numpy(), 'id_portador'].eq(-1).all(),
           'Portadores especiais não apontam para -1.')
    return fato, base


fato, auditoria = construir_fato(dados, dimensoes)
resumo_esperado = auditoria.groupby(['id_mes_extrato', 'id_orgao'], as_index=False).agg(
    transacoes=('valor_centavos', 'size'), centavos=('valor_centavos', 'sum'))
display(resumo_esperado.head())
print('Linhas prontas para carga:', len(fato))

## 6. Carga completa e validação no SQLite
Criamos as tabelas a partir do DDL existente, sem `to_sql(..., if_exists="replace")`, que removeria as restrições. Os índices explícitos são temporariamente removidos do banco novo e recriados após a carga.

A inserção usa parâmetros e uma transação. Validamos contagens, chaves estrangeiras, integridade, valores por registro e totais por mês/órgão. **Nenhuma transformação de negócio ocorre no SQL**: ele recebe as tabelas prontas.

O arquivo temporário fica no mesmo diretório do destino e é fechado antes da substituição. Se qualquer etapa falhar, o banco anterior permanece intacto. Feche conexões externas com o banco antes de executar esta célula. No modo completo, uma execução bem-sucedida substitui `data/database/cpgf_dw.sqlite`.

In [ ]:
def inserir_dataframe(conexao, tabela, df):
    # Nomes vêm exclusivamente dos DataFrames e tabelas definidos neste notebook.
    colunas = ', '.join(f'"{c}"' for c in df.columns)
    parametros = ', '.join('?' for _ in df.columns)
    sql = f'INSERT INTO "{tabela}" ({colunas}) VALUES ({parametros})'
    registros = df.astype(object).where(df.notna(), None).itertuples(index=False, name=None)
    conexao.executemany(sql, registros)


def validar_banco(conexao, dimensoes, fato, auditoria):
    for tabela, df in {**dimensoes, 'FATO_TRANSACAO_CPGF': fato}.items():
        quantidade = conexao.execute(f'SELECT COUNT(*) FROM "{tabela}"').fetchone()[0]
        exigir(quantidade == len(df), f'{tabela}: contagem após carga divergente.')
    exigir(not conexao.execute('PRAGMA foreign_key_check').fetchall(), 'Há chaves estrangeiras inválidas.')
    exigir(conexao.execute('PRAGMA integrity_check').fetchall() == [('ok',)], 'Falha na integridade SQLite.')
    # Confere o valor efetivamente armazenado de cada linha, em ordem de inserção.
    centavos_db = conexao.execute(
        'SELECT CAST(ROUND(valor_transacao * 100) AS INTEGER) FROM FATO_TRANSACAO_CPGF ORDER BY id_transacao')
    exigir(all(linha[0] == int(valor) for linha, valor in zip(centavos_db, auditoria['valor_centavos'])),
           'Perda de precisão monetária no armazenamento.')
    obtido = pd.read_sql_query("""
        SELECT id_mes_extrato, id_orgao, COUNT(*) AS transacoes,
               SUM(CAST(ROUND(valor_transacao * 100) AS INTEGER)) AS centavos
        FROM FATO_TRANSACAO_CPGF
        GROUP BY id_mes_extrato, id_orgao
        ORDER BY id_mes_extrato, id_orgao
    """, conexao)
    esperado = auditoria.groupby(['id_mes_extrato', 'id_orgao'], as_index=False).agg(
        transacoes=('valor_centavos', 'size'), centavos=('valor_centavos', 'sum'))
    pd.testing.assert_frame_equal(obtido, esperado, check_dtype=False)
    return obtido


def carregar_dw(destino, ddl, dimensoes, fato, auditoria):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    # Um banco aberto em WAL pode ter estado fora do arquivo principal.
    exigir(not any(Path(str(destino) + sufixo).exists() for sufixo in ('-wal', '-shm', '-journal')),
           'Há arquivos auxiliares do banco destino. Feche as conexões e revise antes da substituição.')
    with tempfile.NamedTemporaryFile(dir=destino.parent, prefix='.cpgf_etl_', suffix='.sqlite', delete=False) as arq:
        temporario = Path(arq.name)
    try:
        conexao = sqlite3.connect(temporario)
        try:
            conexao.execute('PRAGMA foreign_keys = ON')
            conexao.executescript(Path(ddl).read_text(encoding='utf-8'))
            indices = conexao.execute("SELECT name, sql FROM sqlite_master WHERE type='index' AND sql IS NOT NULL").fetchall()
            for nome, _ in indices:
                conexao.execute(f'DROP INDEX "{nome}"')
            with conexao:
                for tabela, df in dimensoes.items():
                    inserir_dataframe(conexao, tabela, df)
                inserir_dataframe(conexao, 'FATO_TRANSACAO_CPGF', fato)
                for _, sql in indices:
                    conexao.execute(sql)
                resumo = validar_banco(conexao, dimensoes, fato, auditoria)
        finally:
            conexao.close()
        temporario.replace(destino)
    finally:
        temporario.unlink(missing_ok=True)
    return resumo


resumo_carregado = carregar_dw(DESTINO, DDL, dimensoes, fato, auditoria)
print('DW carregado e validado:', DESTINO)
print('Transações:', int(resumo_carregado['transacoes'].sum()))
print('Total em centavos:', int(resumo_carregado['centavos'].sum()))
print('Tempo de execução (s):', round(time.perf_counter() - INICIO, 2))
display(resumo_carregado.head())

## 7. Como revisar esta entrega

1. Reinicie o kernel e execute todas as células. Confira o manifesto e os tipos classificados como `Outros`.
2. Confira o resumo: nenhuma transação é descartada e os totais por mês/órgão precisam coincidir antes e depois da carga.
3. Execute novamente com as mesmas entradas: contagens e valores devem ser iguais, sem acumular registros.
4. Para apresentação rápida, use `MODO_AMOSTRA=True`; para a entrega completa, volte para `False` e execute tudo.

**Decisões para evolução:** classificar as descrições em `Outros`, confirmar a interpretação de `INT$` na fonte e discutir centavos inteiros no esquema físico, caso o projeto exija exatidão decimal no armazenamento. Carga incremental e o notebook ELT ficam fora desta entrega.

O notebook é versionado sem saídas para evitar arquivos grandes e exposição desnecessária dos dados. O banco é um artefato gerado; o `.gitignore` atual do projeto não o exclui automaticamente, então não o inclua no commit do notebook.